In [4]:
#carregando base e preparando x e y
import sys
sys.path.append("..")
from src.preparacao import carregar_base, separar_x_y

import pandas as pd
base = carregar_base()
X, y = separar_x_y(base, alvo="y_abastecimento")
print(base.shape, y.mean().round(3))

(4055, 66) 0.723


In [5]:
#imports
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [6]:
#separando treino e teste
corte = int(len(base) * 0.8)
X_dev, X_test = X.iloc[:corte], X.iloc[corte:]
y_dev, y_test = y.iloc[:corte], y.iloc[corte:]

print("Treino:", base["ini"].iloc[0].date(), "a", base["ini"].iloc[corte - 1].date(), f"({corte})")
print("Teste final:    ", base["ini"].iloc[corte].date(), "a", base["ini"].iloc[-1].date(), f"({len(base) - corte})")

Treino: 2026-04-22 a 2026-08-31 (3244)
Teste final:     2026-08-31 a 2026-10-07 (811)


In [7]:
#buscar hiperparametros
cv_tempo = TimeSeriesSplit(n_splits=5)

buscas = {
    "Regressão Logística": (
        make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)),
        {"logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}),
    "Árvore de Decisão": (
        make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=42)),
        {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10],
         "decisiontreeclassifier__min_samples_leaf": [5, 10, 20, 50, 100]}),
    "Random Forest": (
        make_pipeline(SimpleImputer(strategy="median"),
                      RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)),
        {"randomforestclassifier__max_depth": [None, 6, 10, 15],
         "randomforestclassifier__min_samples_leaf": [1, 3, 5, 10, 20],
         "randomforestclassifier__max_features": ["sqrt", 0.3, 0.5]}),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(random_state=42),
        {"learning_rate": [0.02, 0.05, 0.1],
         "max_iter": [100, 200, 400],
         "max_depth": [None, 3, 5],
         "min_samples_leaf": [10, 20, 50],
         "l2_regularization": [0, 0.1, 1.0]}),
}

melhores = {}
for nome, (modelo, grade) in buscas.items():
    busca = RandomizedSearchCV(modelo, grade, n_iter=20, scoring="roc_auc",
                               cv=cv_tempo, random_state=42, n_jobs=-1)
    busca.fit(X_dev, y_dev)
    melhores[nome] = busca.best_estimator_
    print(f"{nome}: AUC na validação = {busca.best_score_:.3f}")
    print("   parâmetros:", busca.best_params_)

c:\Users\davif\Documents\FHO\8º Período\Inteligência Artificial I\IA Preditiva - MC 68 e MC 63\IA-Preditiva\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 7 is smaller than n_iter=20. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Regressão Logística: AUC na validação = 0.793
   parâmetros: {'logisticregression__C': 0.03}
Árvore de Decisão: AUC na validação = 0.794
   parâmetros: {'decisiontreeclassifier__min_samples_leaf': 100, 'decisiontreeclassifier__max_depth': 8}
Random Forest: AUC na validação = 0.823
   parâmetros: {'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__max_depth': 6}
Gradient Boosting: AUC na validação = 0.814
   parâmetros: {'min_samples_leaf': 10, 'max_iter': 100, 'max_depth': 3, 'learning_rate': 0.02, 'l2_regularization': 0}


In [8]:
linhas = []
todos = {"Baseline": DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev), **melhores}
for nome, modelo in todos.items():
    prob = modelo.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    linhas.append({"modelo": nome,
                   "acurácia": accuracy_score(y_test, pred),
                   "precisão": precision_score(y_test, pred, zero_division=0),
                   "recall":   recall_score(y_test, pred),
                   "F1":       f1_score(y_test, pred),
                   "AUC":      roc_auc_score(y_test, prob)})

final = pd.DataFrame(linhas)
final.round(3)

,modelo,acurácia,precisão,recall,F1,AUC
0,Baseline,0.736,0.736,1.000,0.848,0.500
1,Regressão Logística,0.801,0.828,0.921,0.872,0.848
2,Árvore de Decisão,0.806,0.857,0.884,0.871,0.866
3,Random Forest,0.820,0.874,0.883,0.878,0.881
4,Gradient Boosting,0.822,0.869,0.893,0.881,0.881
